# Semantic roundtrip: study analysis

For the full report, set the completed job paths and `STYLE_REPORT_DIR`, restart the
kernel and **Run All**.
SQ1, Thinking and the style report must share the exact unrestricted Direct source.
Figures appear inline and are saved as PNG/PDF, with supporting tables and provenance
in `OUTPUT_DIR`.


For partial analysis, run the setup cells and only the required sections:

- A and C use Direct only. Load Direct in A before running C.
- SQ1 also needs the prompt job and C. SQ2 needs the style report and Direct.
- Thinking and high-illustratability comparisons need Direct and their own job.
- Direct ratings need Direct only. Description-route ratings need B's inputs.
- The combined domain comparison needs Direct and B's inputs.

Plots are saved as each section runs. The final technical tables and full-report
export require all sections. Use a separate `OUTPUT_DIR` for partial analyses.


In [ ]:
import os
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.ticker import MaxNLocator

from semantic_roundtrip.analysis import aggregate_titles, load_job
from semantic_roundtrip.analysis.plotting import (
    bb_heatmaps,
    heatmap,
    interval_plot,
    rating_analysis,
    save_figure,
)
from semantic_roundtrip.analysis.reporting import (
    DOMAINS,
    METRIC,
    PAIRS,
    QG,
    SENSITIVITY_METRICS,
    TEXT,
    annotate,
    difference,
    direct_refinement_tables,
    effects,
    export_tables,
    indirect_contrasts,
    load_direct_supplement,
    load_indirect_study,
    matching_contrasts,
    overall_domain_means,
    route_transition_counts,
    technical_tables,
    write_manifest,
)

# Silence only the known pandas deprecation; data/errors are not suppressed.
warnings.filterwarnings(
    "ignore",
    category=FutureWarning,
    message="The behavior of DataFrame concatenation with empty or all-NA entries is deprecated.*",
)

# Run from the repository root or its notebooks/ directory.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sns.set_theme(
    style="whitegrid",
    context="notebook",
    rc={"pdf.fonttype": 42, "ps.fonttype": 42, "axes.unicode_minus": False},
)
from semantic_roundtrip.analysis.prompt_baseline import (
    load_prompt_baseline,
    plot_prompt_baseline,
    prompt_baseline_tables,
)
from semantic_roundtrip.analysis.style_report import (
    display_style_summary,
    load_style_summary,
)

STUDY_REVISION = "final_v15"
sensitivity_tables = {}


In [ ]:
# Replace the absolute example paths, or set the corresponding environment variables.
# A job path is its directory, not an individual child run or a website URL.
# Metadata-only archives retain databases, frozen configs and provenance.
REQUIRE_IMAGE_FILES = os.getenv("ANALYSIS_METADATA_ONLY", "0") != "1"
DIRECT_JOB = os.getenv("DIRECT_JOB", "/absolute/path/to/direct-job")
INDIRECT_JOB = os.getenv("INDIRECT_JOB", "/absolute/path/to/local-indirect-job")
# Choose one Aqueduct mode: the complete 20-condition job, or both split jobs.
AQUEDUCT_JOB = os.getenv("AQUEDUCT_JOB") or None
AQUEDUCT_INDEPENDENT_JOB = os.getenv("AQUEDUCT_INDEPENDENT_JOB") or None
AQUEDUCT_COMPLETION_JOB = os.getenv("AQUEDUCT_COMPLETION_JOB") or None
THINKING_JOB = os.getenv("THINKING_JOB", "/absolute/path/to/matching-thinking-job")
ILLUSTRATABLE_JOB = os.getenv(
    "ILLUSTRATABLE_JOB", "/absolute/path/to/illustratable-job"
)
PROMPT_BASELINE_JOB = os.getenv("PROMPT_BASELINE_JOB", "/absolute/path/to/final-direct-prompt-only")
STYLE_REPORT_DIR = os.getenv("STYLE_REPORT_DIR", "/absolute/path/to/completed-style-report")
OUTPUT_DIR = (
    Path(os.getenv("OUTPUT_DIR", ROOT / "notebooks/results/final"))
    .expanduser()
    .resolve()
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Reading the figures

Primary accuracy uses the prompt check, title-aware image verification and Strict Exact
Match, with no image check for prompt-only input. Rejected, failed and missing outcomes
score zero over all planned observations.

Average four observations per image-route title and two per prompt-route title.
Pointwise 95% intervals use 10,000 paired whole-title bootstrap samples within domains
(seed 20260829). Accuracy is in percent and differences are in percentage points (pp).
Blind-strict verification and normalized matching are exported separately.


## A: RQ1 Direct model configurations

### Load once

Run this cell for the 4×4 analysis and the paired-route comparison. No indirect jobs are needed.

In [ ]:
direct = load_job(DIRECT_JOB, require_image_files=REQUIRE_IMAGE_FILES)
direct_obs = annotate(direct.observations)
direct_titles = aggregate_titles(direct_obs, condition_columns=["pg", "bb", "bi"])
direct_only = direct_titles[direct_titles.route == "direct"]

### Family, generation and PG/TG combination

The 4×4 matrix and four planned model contrasts separate PG, TG and same-versus-mixed
pairing, where the last is a PG×TG interaction. TG comparisons reuse images, whereas
PG comparisons change generated images.
The exploratory four TG-minus-PG and six marginal-TG comparisons use separate
approximate familywise 95% Bonferroni-percentile intervals alongside pointwise intervals, without
establishing general role importance or model ranks.

In [ ]:
direct_cells = 100 * direct_only.groupby(["pg", "bi"])[METRIC].mean().unstack().reindex(
    index=QG, columns=QG
)
fig, ax = plt.subplots(figsize=(6, 5), layout="constrained")
fig.colorbar(
    heatmap(ax, direct_only, QG, ""), ax=ax, label="End-to-end Strict Exact Match (%)"
)
save_figure(
    fig, OUTPUT_DIR / "direct_4x4", "RQ1: Direct reconstruction across PG and TG models"
)
blind_strict_cells = 100 * direct_only.groupby(["pg", "bi"])[
    "end_to_end_strict_accuracy"
].mean().unstack().reindex(index=QG, columns=QG)
sensitivity_tables["blind_strict_direct_cells"] = blind_strict_cells.reset_index()


In [ ]:
direct_contrasts = {}
for label, (x, y) in PAIRS.items():
    xx, xy, yx, yy = [
        f"direct_pg_{pg}_bi_{bi}" for pg, bi in [(x, x), (x, y), (y, x), (y, y)]
    ]
    pair_label = f"{label} ({y.upper()} − {x.upper()})"
    direct_contrasts[f"PG: {pair_label}"] = difference([yx, yy], [xx, xy])
    direct_contrasts[f"BI: {pair_label}"] = difference([xy, yy], [xx, yx])
    direct_contrasts[f"Same − mixed: {label}"] = difference([xx, yy], [xy, yx])
direct_effects = effects(direct_only, direct_contrasts)
roles = ["PG", "BI", "Same − mixed"]
panel_titles = [
    "Prompt generation (PG)",
    "Title guessing (TG)",
    "Model pairing (same - mixed)",
]
comparison_colors = {
    "Qwen": "#0072B2",
    "Gemma": "#D55E00",
    "Family 2025": "#009E73",
    "Family 2026": "#CC79A7",
}
overall = direct_effects[direct_effects.domain == "all"]
x_min = 5 * np.floor((overall.ci95_low.min() - 3) / 5)
x_max = 5 * np.ceil((overall.ci95_high.max() + 3) / 5)
fig, axes = plt.subplots(1, 3, figsize=(14, 4.8), sharex=True, layout="constrained")
for role, title, ax in zip(roles, panel_titles, axes):
    table = direct_effects[
        (direct_effects.domain == "all")
        & direct_effects.comparison.str.startswith(role + ":")
    ].copy()
    table["comparison"] = table.comparison.str.removeprefix(role + ": ").str.replace(
        "−", "-", regex=False
    )
    interval_plot(ax, table, colors=list(comparison_colors.values()))
    ax.set(title=title, xlim=(x_min, x_max))
fig.supxlabel(
    "Positive PG/TG values favour the first model in each subtraction; positive pairing values favour same-model pairs.",
    fontsize=9,
    color="#444444",
)
save_figure(
    fig,
    OUTPUT_DIR / "direct_primary_effects",
    "RQ1: Direct reconstruction - planned model contrasts",
)
sensitivity_tables["sensitivity_direct_effects"] = pd.concat([
    effects(direct_only, direct_contrasts, metric=m).assign(metric=m)
    for m in SENSITIVITY_METRICS
], ignore_index=True)
sensitivity_tables["blind_strict_direct_effects"] = (
    sensitivity_tables["sensitivity_direct_effects"]
    .query("metric == 'end_to_end_strict_accuracy'")
    .drop(columns="metric")
    .reset_index(drop=True)
)


In [ ]:
exploratory_direct = direct_refinement_tables(direct_only)


## B: RQ2 Description-mediated model configurations

### Load once

Load the completed local 16-condition job and either the complete 20-condition Aqueduct
job or both split jobs (12 independent + 8 completion). Do not mix the two input modes.
The loader validates the complete
matrix, shared settings, frozen prompts/workflows and every source binding.

In [ ]:
indirect_jobs, full_obs, indirect_job_paths = load_indirect_study(
    INDIRECT_JOB, aqueduct_path=AQUEDUCT_JOB,
    independent_path=AQUEDUCT_INDEPENDENT_JOB,
    completion_path=AQUEDUCT_COMPLETION_JOB,
    require_image_files=REQUIRE_IMAGE_FILES,
)
aqueduct_mode = "monolithic_20"
if AQUEDUCT_INDEPENDENT_JOB:
    independent_cells = indirect_jobs["Aqueduct independent"].observations.condition.nunique()
    completion_cells = indirect_jobs["Aqueduct completion"].observations.condition.nunique()
    aqueduct_mode = f"split_{independent_cells}_plus_{completion_cells}"
local_obs = annotate(indirect_jobs["Indirect"].observations)
local_titles = aggregate_titles(local_obs, condition_columns=["pg", "bb", "bi"])
full_titles = aggregate_titles(full_obs, condition_columns=["pg", "bb", "bi"])
full_ratings = pd.concat([job.ratings for job in indirect_jobs.values()], ignore_index=True)

### PG/ID/TG combinations

Four ID panels show the complete 3×4×3 matrix, followed by six planned local D32/O120
contrasts. Two hosted contrasts compare V4 with the D32/O120 mean over the full matrix.
These compare deployed configurations, not isolated architecture or age effects.

In [ ]:
bb_heatmaps(
    full_titles,
    TEXT,
    OUTPUT_DIR / "indirect_3x4x3_V4_hosted",
    "RQ2: Description-mediated reconstruction (V4 externally hosted)",
)

In [ ]:
local_effects = effects(local_titles, indirect_contrasts(local_titles)).assign(
    scope="D32/O120 submatrix"
)
hosted_contrasts = {
    name: weights
    for name, weights in indirect_contrasts(full_titles).items()
    if "V4" in name
}
hosted_effects = effects(full_titles, hosted_contrasts).assign(
    scope="Full 3×4×3 matrix"
)
indirect_effects = pd.concat([local_effects, hosted_effects], ignore_index=True)
plot_effects = indirect_effects[indirect_effects.domain == "all"].copy()
plot_effects["comparison"] = plot_effects.scope + ": " + plot_effects.comparison
fig, ax = plt.subplots(figsize=(9, 5), layout="constrained")
interval_plot(ax, plot_effects)
save_figure(
    fig,
    OUTPUT_DIR / "indirect_primary_effects",
    "RQ2: Description-mediated reconstruction - model-role contrasts",
)
sensitivity_tables["sensitivity_indirect_effects"] = pd.concat([
    pd.concat([
        effects(local_titles, indirect_contrasts(local_titles), metric=m)
        .assign(scope="D32/O120 submatrix"),
        effects(full_titles, hosted_contrasts, metric=m)
        .assign(scope="Full 3×4×3 matrix"),
    ], ignore_index=True).assign(metric=m)
    for m in SENSITIVITY_METRICS
], ignore_index=True)


In [ ]:
local_matching = effects(local_titles, matching_contrasts(local_titles)).assign(
    scope="D32/O120 submatrix"
)
full_matching = effects(full_titles, matching_contrasts(full_titles)).assign(
    scope="Full 3×4×3 matrix"
)
indirect_matching = pd.concat([local_matching, full_matching], ignore_index=True)
sensitivity_tables["sensitivity_indirect_matching"] = pd.concat([
    pd.concat([
        effects(local_titles, matching_contrasts(local_titles), metric=m)
        .assign(scope="D32/O120 submatrix"),
        effects(full_titles, matching_contrasts(full_titles), metric=m)
        .assign(scope="Full 3×4×3 matrix"),
    ], ignore_index=True).assign(metric=m)
    for m in SENSITIVITY_METRICS
], ignore_index=True)


## C: RQ3 Paired reconstruction routes

Compare direct and description-mediated recovery from the same images in the four
ID=TG diagonal conditions and their equal-weight mean. Transition counts describe
images, while intervals use titles. Route differences do not isolate information loss.

In [ ]:
diagonal = direct_titles[direct_titles.pg == direct_titles.bi]
route_means = (
    100 * diagonal.groupby(["pg", "route"])[METRIC].mean().unstack()
).reindex(QG)
route_scores = diagonal.assign(
    condition_route=diagonal.condition + "__" + diagonal.route
)
route_contrasts = {
    m.upper(): difference(
        [f"direct_pg_{m}_bi_{m}__description"], [f"direct_pg_{m}_bi_{m}__direct"]
    )
    for m in QG
}
route_contrasts["Mean of four baselines"] = difference(
    [f"direct_pg_{m}_bi_{m}__description" for m in QG],
    [f"direct_pg_{m}_bi_{m}__direct" for m in QG],
)
route_effects = effects(
    route_scores, route_contrasts, condition_column="condition_route"
)
fig, axes = plt.subplots(1, 2, figsize=(12, 4), layout="constrained")
for route, marker, offset in [("direct", "o", -0.08), ("description", "s", 0.08)]:
    axes[0].plot(
        np.arange(4) + offset,
        route_means[route],
        marker=marker,
        linestyle="none",
        label=route,
    )
axes[0].set(
    xticks=range(4),
    xticklabels=[m.upper() for m in QG],
    xlabel="Baseline (PG=ID=TG)",
    ylabel="End-to-end Strict Exact Match (%)",
    ylim=(-2, 102),
)
axes[0].legend()
interval_plot(axes[1], route_effects[route_effects.domain == "all"])
axes[0].set_title("Absolute accuracy")
axes[1].set_title("Description - direct")
save_figure(
    fig,
    OUTPUT_DIR / "paired_routes",
    "RQ3: Direct and description-mediated reconstruction",
)
route_transitions = route_transition_counts(direct_obs)
sensitivity_tables["blind_strict_route_transitions"] = route_transition_counts(
    direct_obs, score="end_to_end_strict_score"
).reset_index()
sensitivity_tables["sensitivity_paired_routes"] = pd.concat([
    effects(route_scores, route_contrasts, condition_column="condition_route", metric=m)
    .assign(metric=m) for m in SENSITIVITY_METRICS
], ignore_index=True)
sensitivity_tables["blind_strict_route_means_percent"] = (
    100 * diagonal.groupby(["pg", "route"])["end_to_end_strict_accuracy"].mean().unstack()
).reindex(QG).reset_index()


## D: Secondary analyses

The following sections use the prompt, style, Thinking and high-illustratability inputs
configured above, without replacing the main random-title baseline.


### SQ1: Prompt and image reconstruction

All sixteen cells reuse the unrestricted source prompts, images and image predictions,
adding 2,880 prompt guesses (180 per cell). The four-diagonal three-route subset reuses
720 of these guesses and 1,440 observations per image route, including the RQ3 evidence.
Differences describe route-level recovery, not a guaranteed prompt upper bound or the
location of semantic loss.


In [ ]:
sq1, sq1_obs, sq1_titles = load_prompt_baseline(
    PROMPT_BASELINE_JOB, DIRECT_JOB, require_image_files=REQUIRE_IMAGE_FILES
)
sq1_tables = prompt_baseline_tables(sq1_obs, sq1_titles)
sq1_tables["sq1_rq3_reference"] = route_effects.assign(
    evidence="Existing RQ3 Direct/Indirect contrast; reused, not independent replication"
)
sq1_tables.update({
    f"sq1_{name}": table
    for name, table in technical_tables(
        {"SQ1": sq1_obs}, {"SQ1": sq1_titles}, [("SQ1", sq1)]
    ).items()
    if name not in {"error_attempts", "task_time_minutes"}
})
plot_prompt_baseline(sq1_titles, sq1_tables, OUTPUT_DIR)


`sq1_common_valid_inputs.csv` describes inputs passing both the prompt and title-aware
image checks, with missing predictions still zero. It keeps the sixteen-cell and
four-diagonal subsets separate and does not replace full-denominator accuracy.
`blind_strict_sq1_common_valid_inputs.csv` repeats this diagnostic with blind-strict acceptance.


### SQ2: Frozen four-style evidence

Import the validated compact summary from `style_decision.ipynb` without recomputing
its four-style comparison. The source report retains all detailed effects and diagnostics,
using Unrestricted as the reference without automatic style selection.

The validated summary files are copied to `OUTPUT_DIR/style_summary`.


In [ ]:
_, style_provenance = load_style_summary(
    STYLE_REPORT_DIR, DIRECT_JOB, OUTPUT_DIR
)


In [ ]:
display_style_summary(style_provenance["directory"])


### SQ3: Thinking

Compare twelve new cells plus four unchanged Q25/G3 baselines with the matching Direct
job, overall and for Thinking in PG only, TG only or both roles. These groups contain
different model pairs and do not isolate a role interaction. Changed output ceilings
and timeouts make this a deployment-policy comparison, not an isolated thinking effect
or equal test-time compute.

In [ ]:
thinking, thinking_obs, thinking_changed = load_direct_supplement(
    THINKING_JOB, require_image_files=REQUIRE_IMAGE_FILES
)
unchanged = direct_only[
    direct_only.pg.isin(["q25", "g3"]) & direct_only.bi.isin(["q25", "g3"])
]
thinking_direct = pd.concat([unchanged, thinking_changed], ignore_index=True)
if thinking_direct.condition.nunique() != 16:
    raise ValueError(
        "Thinking matrix does not contain 12 changed and 4 baseline cells."
    )
fig, axes = plt.subplots(1, 3, figsize=(15, 4.8), layout="constrained")
image = heatmap(axes[0], direct_only, QG, "Thinking off")
heatmap(axes[1], thinking_direct, QG, "Native thinking for Q38/G4")
delta = heatmap(axes[2], thinking_direct, QG, "Native - off", baseline=direct_only)
fig.colorbar(image, ax=list(axes[:2]), label="End-to-end Strict Exact Match (%)")
fig.colorbar(delta, ax=axes[2], label="Difference (pp)")
save_figure(fig, OUTPUT_DIR / "direct_thinking_matrices", "SQ3: Native thinking")

In [ ]:
off_scores = direct_only.assign(
    condition_mode=lambda f: "off__" + f.condition.astype(str)
)
native_scores = thinking_direct.assign(
    condition_mode=lambda f: "native__" + f.condition.astype(str)
)
combined = pd.concat([off_scores, native_scores], ignore_index=True)
cells = thinking_direct[["condition", "pg", "bi"]].drop_duplicates()
native_pg = cells.pg.isin(["q38", "g4"])
native_bi = cells.bi.isin(["q38", "g4"])
thinking_groups = {
    "All 16 cells": cells.condition,
    "12 changed cells": cells.loc[native_pg | native_bi, "condition"],
    "PG only (4 cells)": cells.loc[native_pg & ~native_bi, "condition"],
    "BI only (4 cells)": cells.loc[~native_pg & native_bi, "condition"],
    "PG and BI (4 cells)": cells.loc[native_pg & native_bi, "condition"],
}
thinking_contrasts = {
    f"{label}: native - off": difference(
        native_scores.loc[
            native_scores.condition.isin(conditions), "condition_mode"
        ].unique(),
        off_scores.loc[
            off_scores.condition.isin(conditions), "condition_mode"
        ].unique(),
    )
    for label, conditions in thinking_groups.items()
}
thinking_effects = effects(
    combined, thinking_contrasts, condition_column="condition_mode"
)
plotted = thinking_effects[thinking_effects.domain == "all"].copy()
plotted["comparison"] = plotted.comparison.str.split(":").str[0]
fig, ax = plt.subplots(figsize=(9, 4.5), layout="constrained")
interval_plot(ax, plotted)
save_figure(
    fig, OUTPUT_DIR / "direct_thinking_effects", "SQ3: Native thinking - thinking off"
)
sensitivity_tables["sensitivity_thinking_effects"] = pd.concat([
    effects(combined, thinking_contrasts, condition_column="condition_mode", metric=m)
    .assign(metric=m) for m in SENSITIVITY_METRICS
], ignore_index=True)


### SQ4: Domain comparison

Run sections A and B first. Domain means are shown side by side, not pooled across designs.

In [ ]:
direct_domains = overall_domain_means(direct_only, "Direct 4×4")
indirect_domains = overall_domain_means(full_titles, "Indirect 3×4×3")
domain_results = pd.concat([direct_domains, indirect_domains], ignore_index=True)
sensitivity_tables["sensitivity_domain_results"] = pd.concat([
    pd.concat([
        overall_domain_means(direct_only, "Direct 4×4", metric=m),
        overall_domain_means(full_titles, "Indirect 3×4×3", metric=m),
    ], ignore_index=True).assign(metric=m) for m in SENSITIVITY_METRICS
], ignore_index=True)


In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5), layout="constrained")
offsets = {"Direct 4×4": -0.1, "Indirect 3×4×3": 0.1}
markers = {"Direct 4×4": "o", "Indirect 3×4×3": "s"}
for design, table in domain_results.groupby("design", sort=False):
    table = table.set_index("domain").reindex(DOMAINS)
    y = np.arange(len(DOMAINS)) + offsets[design]
    errors = np.vstack(
        [table.estimate - table.ci95_low, table.ci95_high - table.estimate]
    )
    ax.errorbar(
        table.estimate, y, xerr=errors, fmt=markers[design], capsize=2, label=design
    )
ax.set(
    yticks=range(len(DOMAINS)),
    yticklabels=list(DOMAINS),
    xlim=(0, 100),
    xlabel="End-to-end Strict Exact Match (%), pointwise 95% CI",
)
ax.legend()
save_figure(
    fig, OUTPUT_DIR / "domain_overview", "SQ4: Reconstruction accuracy by domain"
)

### SQ5: Model-rated illustratability

#### Association on the direct route

Correlations pair each PG model's rating with direct accuracy averaged across all four
TG models. Histograms use one equal Q25/G3/Q38/G4 mean per title, requiring all four
ratings and reporting missing means separately.

In [ ]:
title_characteristics = direct_obs[
    ["dataset_id", "item_key", "domain"]
].drop_duplicates()


In [ ]:
direct_rating_summary, direct_rating_pairs = rating_analysis(
    direct.ratings, direct_only, None, None, plot=False
)
sensitivity_tables["sensitivity_direct_illustratability"] = pd.concat([
    rating_analysis(direct.ratings, direct_only, None, None, metric=m, plot=False)[0]
    .assign(metric=m) for m in SENSITIVITY_METRICS
], ignore_index=True)
direct_rating_summary


In [ ]:
rating_distribution = direct.ratings.reindex(
    columns=["entry_name", "dataset_id", "domain", "item_key", "score"]
).copy()
rating_distribution["pg"] = rating_distribution.entry_name.str.extract(
    "_pg_([^_]+)_", expand=False
)
rating_index = ["dataset_id", "domain", "item_key"]
rating_means = (
    rating_distribution.drop_duplicates([*rating_index, "pg"])
    .pivot(index=rating_index, columns="pg", values="score")
    .reindex(columns=QG)
    .mean(axis=1, skipna=False)
    .rename("mean_score")
)
rating_means = title_characteristics[rating_index].merge(
    rating_means, on=rating_index, how="left"
)
rating_counts = (
    rating_means.groupby("domain")
    .mean_score.agg(titles="size", complete_ratings="count")
    .reindex(DOMAINS)
)
rating_counts["missing_ratings"] = rating_counts.titles - rating_counts.complete_ratings
fig, axes = plt.subplots(
    1, 3, figsize=(12, 3.8), sharex=True, sharey=True, layout="constrained"
)
for domain, ax in zip(DOMAINS, axes):
    sns.histplot(
        data=rating_means[rating_means.domain == domain],
        x="mean_score",
        color=DOMAINS[domain][0],
        bins=np.arange(0, 101, 5),
        ax=ax,
    )
    ax.set(
        title=f"{domain.title()} (n={rating_counts.loc[domain, 'complete_ratings']})",
        xlabel="Mean illustratability (0–100)",
        ylabel="Titles",
        xlim=(0, 100),
    )
    ax.yaxis.set_major_locator(MaxNLocator(integer=True))
save_figure(
    fig,
    OUTPUT_DIR / "illustratability_distribution",
    "SQ5: Model-averaged illustratability by domain",
)

#### Association on the description-mediated route

Relate each PG rating to accuracy averaged over the same four ID × three TG combinations.

In [ ]:
full_rating_summary, full_rating_pairs = rating_analysis(
    full_ratings, full_titles, None, None, plot=False
)
sensitivity_tables["sensitivity_indirect_illustratability"] = pd.concat([
    rating_analysis(full_ratings, full_titles, None, None, metric=m, plot=False)[0]
    .assign(metric=m) for m in SENSITIVITY_METRICS
], ignore_index=True)
full_rating_summary


#### Random versus high-illustratability titles

The two datasets contain different titles. The matrices and differences below are descriptive
and do not estimate a paired, causal or population-wide effect of filtering.

In [ ]:
illustratable, illustratable_obs, illustratable_direct = load_direct_supplement(
    ILLUSTRATABLE_JOB, require_image_files=REQUIRE_IMAGE_FILES
)
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), layout="constrained")
image = heatmap(axes[0], direct_only, QG, "Random titles")
heatmap(axes[1], illustratable_direct, QG, "High illustratability")
delta = heatmap(
    axes[2], illustratable_direct, QG, "Selected - random", baseline=direct_only
)
fig.colorbar(image, ax=list(axes[:2]), label="End-to-end Strict Exact Match (%)")
fig.colorbar(delta, ax=axes[2], label="Difference (pp)")
save_figure(
    fig,
    OUTPUT_DIR / "direct_illustratability_matrices",
    "SQ5: High-illustratability vs random titles (descriptive)",
)
descriptive_dataset_comparison = pd.DataFrame(
    {
        "dataset": ["Random", "High illustratability"],
        "titles": [
            direct_only.item_key.nunique(),
            illustratable_direct.item_key.nunique(),
        ],
        "mean_accuracy_percent": [
            100 * direct_only[METRIC].mean(),
            100 * illustratable_direct[METRIC].mean(),
        ],
    }
)
sensitivity_tables["sensitivity_dataset_comparison"] = pd.concat([
    pd.DataFrame({
        "dataset": ["Random", "High illustratability"],
        "titles": [direct_only.item_key.nunique(), illustratable_direct.item_key.nunique()],
        "mean_accuracy_percent": [100 * direct_only[m].mean(), 100 * illustratable_direct[m].mean()],
        "metric": m,
    }) for m in SENSITIVITY_METRICS
], ignore_index=True)


## E: Technical validity and supporting accuracy

Run the preceding sections first, with counts read as condition/route observations
rather than unique images or model calls. Do not add pooled `all` rows to detail rows.
Thinking baselines and imported SQ1 image routes are not new computation, while style
diagnostics remain in the separate style report and undefined values are `n/a`.


In [ ]:
observation_sets = {"Direct": direct_obs, "Indirect 3×4×3": full_obs}
title_sets = {"Direct": direct_titles, "Indirect 3×4×3": full_titles}
job_sets = [("Direct", direct), *[("Indirect 3×4×3", job) for job in indirect_jobs.values()]]
for label, job, observations, scores in [
    ("Thinking", thinking, thinking_obs, thinking_direct),
    ("Illustratable", illustratable, illustratable_obs, illustratable_direct),
]:
    observations = observations[observations.route.eq("direct")]
    if label == "Thinking":
        baseline = direct_obs[
            direct_obs.route.eq("direct")
            & direct_obs.pg.isin(["q25", "g3"])
            & direct_obs.bi.isin(["q25", "g3"])
        ]
        observations = pd.concat([observations, baseline], ignore_index=True)
    observation_sets[label] = observations
    title_sets[label] = scores
    job_sets.append((label, job))
observation_sets["SQ1 prompt (new)"] = sq1_obs[sq1_obs.route.eq("prompt")]
title_sets["SQ1 prompt (new)"] = sq1_titles[sq1_titles.route.eq("prompt")]
job_sets.append(("SQ1 prompt (new)", sq1))
technical = technical_tables(observation_sets, title_sets, job_sets)
for name, table in technical.items():
    table.to_csv(OUTPUT_DIR / f"{name}.csv", index=False, na_rep="n/a")

## Reproducibility exports

Export figure data, tables and provenance, counting one prompt decision per prompt and
two image decisions per image. Imported errors and timings are deduplicated at their
original source, with durations in task minutes rather than end-to-end wall time.

In [ ]:
export_tables(
    {
        "direct_cells": direct_cells.reset_index(),
        "direct_effects": direct_effects,
        "paired_routes": route_effects,
        "route_transitions": route_transitions.reset_index(),
        "direct_illustratability": direct_rating_summary,
        "direct_illustratability_pairs": direct_rating_pairs,
        "rating_means": rating_means,
        "rating_counts": rating_counts.reset_index(),
        "indirect_effects": indirect_effects,
        "indirect_matching": indirect_matching,
        "indirect_illustratability": full_rating_summary,
        "indirect_illustratability_pairs": full_rating_pairs,
        "domain_results": domain_results,
        "thinking_effects": thinking_effects,
        "dataset_comparison": descriptive_dataset_comparison,
        **sq1_tables,
        **sensitivity_tables,
        **exploratory_direct,
    },
    OUTPUT_DIR,
)
write_manifest(
    ROOT / "notebooks/final_study.ipynb",
    {
        "Direct": DIRECT_JOB,
        **indirect_job_paths,
        "Thinking": THINKING_JOB,
        "Illustratable": ILLUSTRATABLE_JOB,
        "Prompt baseline": PROMPT_BASELINE_JOB,
    },
    OUTPUT_DIR,
    analysis={
        "purpose": "full_study",
        "study_revision": STUDY_REVISION,
        "require_image_files": REQUIRE_IMAGE_FILES,
        "primary_definition": "prompt check + title-aware image check where applicable + Strict Exact Match",
        "sensitivity_metrics": list(SENSITIVITY_METRICS),
        "exploratory_direct": {"tg_minus_pg_family_size": 4, "tg_pairwise_family_size": 6, "intervals": "pointwise and approximate Bonferroni percentile 95%"},
        "aqueduct_mode": aqueduct_mode,
        "seed_observations_per_title": {"direct": 4, "description": 4, "prompt": 2},
        "style_report": style_provenance,
        "sq1": {"source_style": "Unrestricted", "prompt_direct_cells": 16, "three_route_diagonals": 4, "new_planned_predictions": 2880},
        "primary": METRIC,
        "denominator": "all planned observations",
        "bootstrap": {
            "unit": "paired whole title",
            "strata": ["domain"],
            "repetitions": 10000,
            "seed": 20260829,
            "pointwise": True,
        },
    },
)